# Lecture 6 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week06/lecture06_exercise.ipynb`

**Rules:**
1. Use `px` first, then customise with `update_traces` / `update_layout`
2. Colour encodes a meaningful category — not decoration
3. Insight title names the specific finding
4. Consider: would a bar chart be clearer? If yes, use the bar chart

---


In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('../data/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         60  7168.0   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [2]:
# Task 1
import pandas as pd
import plotly.express as px

#Load the dataset
df = pd.read_csv('../data/global_energy_mix.csv')

fossil_sources = ['Coal', 'Oil', 'Natural Gas']

df['Source_Type'] = df['Source'].apply(lambda x: 'Fossil' if x in fossil_sources else 'Other')
fossil_df = df.loc[df['Source_Type'] == 'Fossil'].copy()

region_totals = fossil_df.groupby('Region')['TWh'].sum().sort_values(ascending=False)
country_totals = fossil_df.groupby('Country')['TWh'].sum().sort_values(ascending=False)

top_region = region_totals.index[0]  
top_country = country_totals.index[0]

source_colors = {
    'Coal': '#0072B2',        
    'Oil': '#D55E00',        
    'Natural Gas': '#009E73'
}

#Treemap
fig = px.treemap(
    fossil_df,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map=source_colors,
    title=f"Fossil Fuel Dependency: {top_country} is the largest fossil fuel consumer, with {top_region} leading by Region"
)

fig.update_traces(
    textinfo='label+value',
    hovertemplate='<b>%{label}</b><br>TWh: %{value}<extra></extra>'
)

# Applying the Grey-out Logic for the region and country
labels = fig.data[0].labels
new_colors = []
for label in labels:
    if label in source_colors:
        new_colors.append(source_colors[label])
    else:
        new_colors.append('lightgrey')

fig.data[0].marker.colors = new_colors

fig.update_layout(
    margin=dict(t=80, l=20, r=20, b=20),
    template="plotly_white"
)

fig.show()

## Task 2 — Sunburst: tipping behaviour by day and meal time

**What to build:** A sunburst chart using the built-in `tips` dataset showing how **total bill amount** is distributed across day → time → smoker status.

**Requirements:**
- Load tips with `px.data.tips()`
- Aggregate **total bill** (sum of `total_bill`) per group — not count
- Hierarchy: `path=['day', 'time', 'smoker']`
- Colour encodes smoker status with a CVD-safe blue/orange palette
- Grey out parent nodes (day and time level)
- Use `percent parent` for text labels
- Insight title describing where the most spending happens

> 💡 `tips.groupby(['day', 'time', 'smoker'])['total_bill'].sum().reset_index()`


In [3]:
# Task 2
import plotly.express as px
import pandas as pd

#Load the data
tips = px.data.tips()

tips_agg = tips.groupby(['day', 'time', 'smoker'])['total_bill'].sum().reset_index()

top_row = tips_agg.sort_values('total_bill', ascending=False).iloc[0]
peak_description = f"{top_row['day']} {top_row['time']} ({'Smokers' if top_row['smoker']=='Yes' else 'Non-Smokers'})"


color_map = {
    'Yes': '#E69F00',
    'No':  '#0072B2',
    'parent': '#D3D3D3'
}

# Creating the Sunburst Chart
fig = px.sunburst(
    tips_agg,
    path=['day', 'time', 'smoker'],
    values='total_bill',
    title=f"<b>Total Bill Distribution Analysis</b><br>Peak spending segment: {peak_description}",
    template='plotly_white'
)

fig.data[0].marker.colors = [
    color_map.get(label, color_map['parent']) if ids.count('/') == 2 
    else color_map['parent']
    for ids, label in zip(fig.data[0].ids, fig.data[0].labels)
]

fig.update_traces(
    textinfo='label+percent parent',
    insidetextorientation='radial',
    hovertemplate=(
        '<b>%{label}</b><br>'
        'Total Spent: $%{value:.2f}<br>'
        'Contribution to %{parent}: %{percentParent:.1%}<extra></extra>'
    )
)

fig.update_layout(
    margin=dict(t=80, l=10, r=10, b=10),
    font=dict(family="Arial, sans-serif", size=14),
    title_font_size=20
)

fig.show()


## Task 3 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country


In [6]:
# Task 3 — charts
# Task 3 — charts
import plotly.express as px
import pandas as pd

#Load the data
df = pd.read_csv('../data/global_energy_mix.csv')

# Steps for Filter and Aggregate
low_carbon_df = df[df['Source'].isin(['Nuclear', 'Hydro'])].copy()
agg_df = low_carbon_df.groupby('Country')['TWh'].sum().reset_index()

agg_df = agg_df.sort_values('TWh', ascending=True)

# Dummy root node for treemap
agg_df['All'] = 'Low-carbon'

leading_country = agg_df.iloc[-1]['Country']
leading_twh = agg_df.iloc[-1]['TWh']

fig_tree = px.treemap(
    agg_df, 
    path=['All', 'Country'], 
    values='TWh',
    title='Distribution of Low-carbon Energy (Nuclear + Hydro) by Country',
    color_discrete_sequence=["#E27F6E"]
)
fig_tree.update_traces(textinfo="label+value")

fig_bar = px.bar(
    agg_df,
    x='TWh',
    y='Country',
    orientation='h',
    title=f"Insight: {leading_country} is the Leading Producer of Low-carbon Energy ({leading_twh:,.0f} TWh)",
    labels={'TWh': 'Total Low-carbon TWh', 'Country': ''},
    color_discrete_sequence=["#437E4A"] 
)

fig_bar.update_traces(texttemplate='%{x:.2s}', textposition='outside')
fig_bar.update_layout(
    xaxis_title="Generation (TWh)",
    yaxis={'categoryorder': 'total ascending'},
    template='plotly_white',
    margin=dict(t=80, l=100)
)

fig_tree.show()
fig_bar.show()
